#        **Analyse et nettoyage du dataset BAAC – France 2024**

In [147]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [148]:
caract = pd.read_csv("caract-2024.csv", sep=";", encoding="utf-8", low_memory=False)
lieux = pd.read_csv("lieux-2024.csv", sep=";", encoding="utf-8", low_memory=False)
vehicules = pd.read_csv("vehicules-2024.csv", sep=";", encoding="utf-8", low_memory=False)
usagers = pd.read_csv("usagers-2024.csv", sep=";", encoding="utf-8", low_memory=False)

---
## 1. Nettoyage de la table `caract` (Caractéristiques des accidents)

### 1.1 Exploration initiale

In [149]:
print("Colonnes :", caract.columns.tolist())
print()
caract.info()
display(caract.head())

Colonnes : ['Num_Acc', 'jour', 'mois', 'an', 'hrmn', 'lum', 'dep', 'com', 'agg', 'int', 'atm', 'col', 'adr', 'lat', 'long']

<class 'pandas.DataFrame'>
RangeIndex: 54402 entries, 0 to 54401
Data columns (total 15 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   Num_Acc  54402 non-null  int64
 1   jour     54402 non-null  int64
 2   mois     54402 non-null  int64
 3   an       54402 non-null  int64
 4   hrmn     54402 non-null  str  
 5   lum      54402 non-null  int64
 6   dep      54402 non-null  str  
 7   com      54402 non-null  str  
 8   agg      54402 non-null  int64
 9   int      54402 non-null  int64
 10  atm      54402 non-null  int64
 11  col      54402 non-null  int64
 12  adr      52092 non-null  str  
 13  lat      54402 non-null  str  
 14  long     54402 non-null  str  
dtypes: int64(9), str(6)
memory usage: 6.2 MB


,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
0,202400000001,25,3,2024,07:40,2,70,70285,1,1,5,1,D438,"47,56277000","6,75832000"
1,202400000002,20,3,2024,15:05,1,21,21054,2,3,7,6,HOTEL DIEU (RUE DE L'),"47,02109000","4,83755000"
2,202400000003,22,3,2024,19:30,2,15,15012,1,1,1,6,Allée des Tilleuls,"44,90238400","2,49641800"
3,202400000004,24,3,2024,17:50,1,14,14118,2,3,7,3,128 Rue d'Authie,"49,19166000","-0,39851000"
4,202400000005,25,3,2024,19:35,5,13,13106,1,3,2,5,BEDOULE (CHEMIN DE LA),"43,39000000","5,35000000"


In [150]:
print("Types des attributs :")
display(caract.dtypes)

Types des attributs :


Num_Acc    int64
jour       int64
mois       int64
an         int64
hrmn         str
lum        int64
dep          str
com          str
agg        int64
int        int64
atm        int64
col        int64
adr          str
lat          str
long         str
dtype: object

In [151]:
print("Nombre de doublons :", caract.duplicated().sum())
print()
print("Valeurs manquantes par colonne :")
display(caract.isna().sum())

Nombre de doublons : 0

Valeurs manquantes par colonne :


Num_Acc       0
jour          0
mois          0
an            0
hrmn          0
lum           0
dep           0
com           0
agg           0
int           0
atm           0
col           0
adr        2310
lat           0
long          0
dtype: int64

### 1.2 Nettoyage de la table `caract`

Aucun doublon détecté. La colonne `adr` présente des valeurs manquantes : les cases vides sont remplacées par `"Unknown"` pour indiquer une adresse non renseignée.

In [152]:
caract["adr"] = caract["adr"].fillna("Unknown")
print("Valeurs manquantes restantes dans 'adr' :", caract["adr"].isna().sum())

Valeurs manquantes restantes dans 'adr' : 0


Les colonnes codées utilisent `-1` comme valeur "non renseigné" dans la nomenclature BAAC (ex. `atm`, `col`, `lum`). Ces valeurs ne correspondent à aucune catégorie valide et sont remplacées par `NaN` pour ne pas biaiser les analyses statistiques.

In [153]:
cols_coded_caract = ["atm", "col", "lum"]
for col in cols_coded_caract:
    caract[col] = caract[col].replace(-1, np.nan)

print("Valeurs -1 remplacées par NaN dans :", cols_coded_caract)

Valeurs -1 remplacées par NaN dans : ['atm', 'col', 'lum']


Vérification des valeurs aberrantes via `describe()` :

In [155]:
display(caract.describe(include='all'))

,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long
count,5.440200e+04,54402.000000,54402.000000,54402.0,54402,54402.000000,54402,54402,54402.000000,54402.000000,54402.000000,54396.000000,54402,54402,54402
unique,NaN,NaN,NaN,NaN,1414,NaN,107,11285,NaN,NaN,NaN,NaN,28872,52471,52742
top,NaN,NaN,NaN,NaN,18:00,NaN,75,97302,NaN,NaN,NaN,NaN,Unknown,"48,91936344","5,40000000"
freq,NaN,NaN,NaN,NaN,677,NaN,4191,479,NaN,NaN,NaN,NaN,2310,8,9
mean,2.024000e+11,15.665490,6.615749,2024.0,NaN,1.949708,NaN,NaN,1.625161,2.108195,1.677493,4.018623,NaN,NaN,NaN
std,1.570465e+04,8.737761,3.361701,0.0,NaN,1.488462,NaN,NaN,0.484086,2.042602,1.736956,1.983970,NaN,NaN,NaN
min,2.024000e+11,1.000000,1.000000,2024.0,NaN,1.000000,NaN,NaN,1.000000,1.000000,1.000000,1.000000,NaN,NaN,NaN
25%,2.024000e+11,8.000000,4.000000,2024.0,NaN,1.000000,NaN,NaN,1.000000,1.000000,1.000000,3.000000,NaN,NaN,NaN
50%,2.024000e+11,16.000000,7.000000,2024.0,NaN,1.000000,NaN,NaN,2.000000,1.000000,1.000000,3.000000,NaN,NaN,NaN
75%,2.024000e+11,23.000000,10.000000,2024.0,NaN,3.000000,NaN,NaN,2.000000,2.000000,1.000000,6.000000,NaN,NaN,NaN


Aucune valeur aberrante détectée en dehors des `-1` déjà traités. Les colonnes `dep` et `com` sont conservées en texte (`str`) pour préserver les zéros initiaux (ex. `075` → `75` si converti en entier). La colonne `hrmn` est également conservée en texte car elle représente un code horaire HHMM.

Création d'une colonne `date` à partir des colonnes `jour`, `mois` et `an`, puis suppression de ces dernières :

In [158]:
caract["date"] = pd.to_datetime({
    "year":  caract["an"],
    "month": caract["mois"],
    "day":   caract["jour"],
})
caract.drop(["jour", "mois", "an"], axis=1, inplace=True)
display(caract.head())

,Num_Acc,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long,date
0,202400000001,07:40,2,70,70285,1,1,5,1.0,D438,"47,56277000","6,75832000",2024-03-25
1,202400000002,15:05,1,21,21054,2,3,7,6.0,HOTEL DIEU (RUE DE L'),"47,02109000","4,83755000",2024-03-20
2,202400000003,19:30,2,15,15012,1,1,1,6.0,Allée des Tilleuls,"44,90238400","2,49641800",2024-03-22
3,202400000004,17:50,1,14,14118,2,3,7,3.0,128 Rue d'Authie,"49,19166000","-0,39851000",2024-03-24
4,202400000005,19:35,5,13,13106,1,3,2,5.0,BEDOULE (CHEMIN DE LA),"43,39000000","5,35000000",2024-03-25


Les colonnes `lat` et `long` sont importées en texte (virgule décimale). Elles sont converties en `float` après remplacement de la virgule par un point. Les valeurs non convertibles (cellules vides) deviennent automatiquement `NaN`.

In [159]:
caract["lat"]  = pd.to_numeric(caract["lat"].str.replace(",", ".", regex=False),  errors="coerce")
caract["long"] = pd.to_numeric(caract["long"].str.replace(",", ".", regex=False), errors="coerce")
print("Types après conversion :")
print(caract[["lat", "long"]].dtypes)

Types après conversion :
lat     float64
long    float64
dtype: object


---
## 2. Nettoyage de la table `lieux`

**2.1 Exploration Initiale de la table `lieux`**


In [94]:
print("Les colonnes de la table lieux :\n")
print(lieux.columns)
print("\n")
print("Les différentes infos de la table :\n")
print(lieux.info)
print("\n")
print("Apperçu de la table : \n")
lieux.head()

Les colonnes de la table lieux :

Index(['Num_Acc', 'catr', 'voie', 'v1', 'v2', 'circ', 'nbv', 'vosp', 'prof',
       'pr', 'pr1', 'plan', 'lartpc', 'larrout', 'surf', 'infra', 'situ',
       'vma'],
      dtype='str')


Les différentes infos de la table :

<bound method DataFrame.info of             Num_Acc  catr                                     voie  v1   v2  \
0      202400000001     3                                     D438   0  NaN   
1      202400000002     4                   HOTEL DIEU (RUE DE L')   0  NaN   
2      202400000002     4                            POTERNE (RUE)   0  NaN   
3      202400000003     4                     TILLEULS (ALLEE DES)   0  NaN   
4      202400000004     4       AUTHIE (N° 106 PAIRS -115 IMPAIRS)   0  NaN   
...             ...   ...                                      ...  ..  ...   
70243  202400054398     3                                      NaN  -1  NaN   
70244  202400054399     3                        RUE PIERRE GAUDIN   0  NaN   

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
0,202400000001,3,D438,0,NaN,2,2,0,1,1,260,2,NaN,7,1,0,1,90
1,202400000002,4,HOTEL DIEU (RUE DE L'),0,NaN,2,2,0,1,-1,-1,1,NaN,-1,9,0,1,30
2,202400000002,4,POTERNE (RUE),0,NaN,1,1,0,1,-1,-1,1,NaN,-1,9,0,1,30
3,202400000003,4,TILLEULS (ALLEE DES),0,NaN,2,2,0,1,-1,-1,1,NaN,-1,1,0,3,50
4,202400000004,4,AUTHIE (N° 106 PAIRS -115 IMPAIRS),0,NaN,2,4,0,1,-1,-1,1,NaN,-1,1,9,1,50


In [95]:
print("Le nombre de cellule vide dans chaque colonne : \n")
print(lieux.isna().sum())
print("\n")
print("Le nombre de doublons dans la table :", +lieux.duplicated().sum())
print("\n")
print("Visuel des lignes doublées :")
lieux[lieux.duplicated(keep=False)]


Le nombre de cellule vide dans chaque colonne : 

Num_Acc        0
catr           0
voie       13331
v1             0
v2         64332
circ           0
nbv            0
vosp           0
prof           0
pr             0
pr1            0
plan           0
lartpc     70215
larrout        0
surf           0
infra          0
situ           0
vma            0
dtype: int64


Le nombre de doublons dans la table : 2


Visuel des lignes doublées :


,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
15884,202400012279,3,NaN,-1,NaN,2,-1,-1,2,-1,-1,1,NaN,-1,1,0,1,-1
15885,202400012279,3,NaN,-1,NaN,2,-1,-1,2,-1,-1,1,NaN,-1,1,0,1,-1
57342,202400044389,4,NaN,-1,NaN,2,-1,-1,1,-1,-1,1,NaN,-1,1,0,1,-1
57343,202400044389,4,NaN,-1,NaN,2,-1,-1,1,-1,-1,1,NaN,-1,1,0,1,-1


Les doublons observés dans la table **lieux** correspondent à des lignes strictement identiques pour un même numéro d’accident. Comme chaque accident ne doit être associé qu’à un seul lieu, ces répétitions traduisent une **erreur de saisie ou d’importation** plutôt qu’une situation réelle.
Il y a également des cellules vide notamment dans **lartpc** **v2** et **voie**


In [96]:
lieux.dtypes

Num_Acc    int64
catr       int64
voie         str
v1         int64
v2           str
circ       int64
nbv          str
vosp       int64
prof       int64
pr           str
pr1          str
plan       int64
lartpc       str
larrout      str
surf       int64
infra      int64
situ       int64
vma        int64
dtype: object

**2.1 Début nettoyage de la `lieux`**

Correction des valeurs manquantes en ajoutant `Unknown`

In [97]:
lieux["voie"] = lieux["voie"].fillna("Unknown")
lieux["v2"] = lieux["v2"].fillna("Unknown")
lieux["lartpc"] = lieux["lartpc"].fillna("Unknown")

print("Nombre de valeurs manquantes après correction : \n")
print(lieux.isna().sum())


Nombre de valeurs manquantes après correction : 

Num_Acc    0
catr       0
voie       0
v1         0
v2         0
circ       0
nbv        0
vosp       0
prof       0
pr         0
pr1        0
plan       0
lartpc     0
larrout    0
surf       0
infra      0
situ       0
vma        0
dtype: int64


Pour les attributs `voie`, `v2` et `lartpc`  les cellules vides ont été remplacées par la valeur `Unknown`. Comme il s’agit d’attributs catégoriel et non numérique, cette substitution est cohérente : elle permet d’indiquer explicitement que le type de voie n’est pas renseigné, sans introduire de valeur artificielle.

Suppresion des doublons

In [98]:
lieux = lieux.drop_duplicates(lieux) # Suppresion des doublons
print("Nombre de ligne doublés après corréction :", +lieux.duplicated().sum())

Nombre de ligne doublés après corréction : 0


In [99]:
lieux.describe(include='all')

,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
count,7.024600e+04,70246.000000,70246,70246.000000,70246,70246.000000,70246,70246.000000,70246.000000,70246,70246,70246.000000,70246,70246,70246.000000,70246.000000,70246.000000,70246.000000
unique,NaN,NaN,19751,NaN,28,NaN,15,NaN,NaN,461,1353,NaN,16,65,NaN,NaN,NaN,NaN
top,NaN,NaN,Unknown,NaN,Unknown,NaN,2,NaN,NaN,-1,-1,NaN,Unknown,-1,NaN,NaN,NaN,NaN
freq,NaN,NaN,13329,NaN,64330,NaN,42693,NaN,NaN,27362,27430,NaN,70213,48644,NaN,NaN,NaN,NaN
mean,2.024000e+11,3.401574,NaN,-0.228255,NaN,1.735615,NaN,0.185904,1.215699,NaN,NaN,1.278037,NaN,NaN,1.283660,0.847934,1.640150,53.713037
std,1.570886e+04,1.123817,NaN,0.432342,NaN,0.908136,NaN,0.745741,0.527899,NaN,NaN,0.656018,NaN,NaN,0.808311,2.201933,1.676752,27.237056
min,2.024000e+11,1.000000,NaN,-1.000000,NaN,-1.000000,NaN,-1.000000,-1.000000,NaN,NaN,-1.000000,NaN,NaN,-1.000000,-1.000000,-1.000000,-1.000000
25%,2.024000e+11,3.000000,NaN,0.000000,NaN,1.000000,NaN,0.000000,1.000000,NaN,NaN,1.000000,NaN,NaN,1.000000,0.000000,1.000000,30.000000
50%,2.024000e+11,3.000000,NaN,0.000000,NaN,2.000000,NaN,0.000000,1.000000,NaN,NaN,1.000000,NaN,NaN,1.000000,0.000000,1.000000,50.000000
75%,2.024000e+11,4.000000,NaN,0.000000,NaN,2.000000,NaN,0.000000,1.000000,NaN,NaN,1.000000,NaN,NaN,1.000000,0.000000,1.000000,70.000000


L’analyse statistique des attributs de la table `lieux` révèle aucune valeur aberrante. Les valeurs extrêmes observées (ex. -1 ou 900) correspondent à des codes BAAC indiquant des données non renseignées ou sans objet, et sont donc cohérentes avec la nomenclature officielle.


In [100]:
lieux["lartpc"].unique()

<StringArray>
['Unknown',       '7',       '2',      '10',       '3',     '3,5',       '9',
       '4',      '14',     '4,5',       '5',     '2,5',     '8,5',       '1',
      '12',     '6,2']
Length: 16, dtype: str

On remarque de l'attribut `lartpc` peut etre convertit en int pour faciliter l'analyse après

In [101]:
lieux["lartpc"] = (
    lieux["lartpc"]
    .replace("Unknown", np.nan)
    .str.replace(",", ".", regex=False)
    .replace("", np.nan)
    .astype(float)
)

print("Nouveau type de l'attribut lartpc :", lieux["lartpc"].dtypes)


Nouveau type de l'attribut lartpc : float64


#### **3) Nettoyage de la table `usagers` (Caractéristique des accidents)**

**3.1 Exploration Initiale de la `usagers`**

In [102]:
print("Les colonnes de la table liusagers :\n")
print(usagers.columns)
print("\n")
print("Les différentes infos de la table :\n")
print(usagers.info)
print("\n")
print("Apperçu de la table : \n")
usagers.head()

Les colonnes de la table liusagers :

Index(['Num_Acc', 'id_usager', 'id_vehicule', 'num_veh', 'place', 'catu',
       'grav', 'sexe', 'an_nais', 'trajet', 'secu1', 'secu2', 'secu3', 'locp',
       'actp', 'etatp'],
      dtype='str')


Les différentes infos de la table :

<bound method DataFrame.info of              Num_Acc    id_usager  id_vehicule num_veh  place  catu  grav  \
0       202400000001  203 988 581  155 781 758     A01      1     1     3   
1       202400000001  203 988 582  155 781 759     B01      1     1     1   
2       202400000002  203 988 579  155 781 757     A01     10     3     3   
3       202400000002  203 988 580  155 781 757     A01      1     1     1   
4       202400000003  203 988 574  155 781 756     A01      2     2     4   
...              ...          ...          ...     ...    ...   ...   ...   
125182  202400054401  203 859 570  155 686 119     Y01      1     1     4   
125183  202400054401  203 859 572  155 686 120     A01      1     1     1   
1

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
0,202400000001,203 988 581,155 781 758,A01,1,1,3,1,2003.0,2,1,-1,-1,-1,-1,-1
1,202400000001,203 988 582,155 781 759,B01,1,1,1,1,1997.0,4,1,-1,-1,-1,-1,-1
2,202400000002,203 988 579,155 781 757,A01,10,3,3,2,1927.0,5,0,-1,-1,3,3,1
3,202400000002,203 988 580,155 781 757,A01,1,1,1,1,1987.0,4,1,0,-1,3,3,1
4,202400000003,203 988 574,155 781 756,A01,2,2,4,2,2007.0,5,8,0,-1,-1,-1,-1


In [103]:
print("Le nombre de cellule vide dans chaque colonne : \n")
print(usagers.isna().sum())
print("\n")
print("Le nombre de doublons dans la table :", +usagers.duplicated().sum())


Le nombre de cellule vide dans chaque colonne : 

Num_Acc           0
id_usager         0
id_vehicule       0
num_veh           0
place             0
catu              0
grav              0
sexe              0
an_nais        2579
trajet            0
secu1             0
secu2             0
secu3             0
locp              0
actp              0
etatp             0
dtype: int64


Le nombre de doublons dans la table : 0


In [104]:
usagers["an_nais"].unique()

array([2003., 1997., 1927., 1987., 2007., 2008., 1963., 1981., 2005.,
       2000., 1975., 1983., 1984., 2001., 1996., 1959., 2006., 1979.,
       2012., 2009., 2014., 1965., 1993., 2010., 1958., 1998., 1988.,
       1968., 1976., 1970., 1982., 1999., 2002., 1948., 1946., 1991.,
       1990., 1989., 1995., 2004., 1949., 2023., 1992., 1939., 1977.,
       1964., 1954., 1980., 1966., 1962., 1956., 2020., 1986., 1955.,
       1960., 2015., 2016., 1973., 1978., 1974., 1994., 1971., 1969.,
       1950., 1944., 1985., 1937., 1940., 1947., 1938., 1952., 1961.,
       1972.,   nan, 1957., 2013., 1967., 1934., 1943., 2019., 1953.,
       1933., 1941., 1951., 2011., 1942., 1935., 2021., 1931., 2018.,
       2022., 2017., 1945., 1936., 2024., 1926., 1930., 1932., 1929.,
       1928., 1923., 1924., 1915., 1925., 1914., 1922.])

**Début nettoyage de la table `usagers`**

On peut convertir le type de la colonne en Int


In [105]:
usagers["an_nais"] = usagers["an_nais"].astype("Int64")

print("Nouveau type de l'attribut an_nais :", usagers["an_nais"].dtypes)

Nouveau type de l'attribut an_nais : Int64


Nous laissons les cases vides sous forme de valeurs manquantes (NaN), car elles ne contiennent aucune information exploitable et leur conversion n’apporterait rien à l’analyse. Le type Int64 permet de conserver ces valeurs manquantes sans erreur tout en travaillant avec une colonne d’entiers.

In [106]:
usagers.describe(include='all')


,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
count,1.251870e+05,125187,125187,125187,125187.000000,125187.000000,125187.000000,125187.000000,122608.0,125187.000000,125187.000000,125187.000000,125187.000000,125187.000000,125187,125187.000000
unique,NaN,125187,92654,45,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,13,NaN
top,NaN,203 988 581,155 721 557,A01,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,-1,NaN
freq,NaN,1,56,75443,NaN,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN,61726,NaN
mean,2.024000e+11,NaN,NaN,NaN,2.100745,1.335554,2.524208,1.272696,1985.075411,3.069009,1.938788,0.843906,-0.800994,-0.223002,NaN,-0.814605
std,1.567431e+04,NaN,NaN,NaN,2.584540,0.610862,1.374424,0.559575,19.327486,2.762129,2.384028,2.910122,1.067398,1.276891,NaN,0.638087
min,2.024000e+11,NaN,NaN,NaN,-1.000000,1.000000,1.000000,-1.000000,1914.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,NaN,-1.000000
25%,2.024000e+11,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.000000,1971.0,0.000000,1.000000,-1.000000,-1.000000,-1.000000,NaN,-1.000000
50%,2.024000e+11,NaN,NaN,NaN,1.000000,1.000000,3.000000,1.000000,1988.0,4.000000,1.000000,0.000000,-1.000000,0.000000,NaN,-1.000000
75%,2.024000e+11,NaN,NaN,NaN,2.000000,2.000000,4.000000,2.000000,2001.0,5.000000,2.000000,0.000000,-1.000000,0.000000,NaN,-1.000000


Afin de corriger la présence de la valeur aberrante -1 dans la variable `sexe`, nous avons converti cette colonne en type entier (Int64). Cette conversion permet d’éliminer les valeurs invalides et de ne conserver que les codes valides 1 (homme) et 2 (femme), garantissant ainsi la cohérence des données.

In [111]:
usagers["sexe"] = usagers["sexe"].replace(-1, np.nan)     # Remplacement des -1 par nan
print(usagers["sexe"].unique())

<IntegerArray>
[1, 2, <NA>]
Length: 3, dtype: Int64


La variable `place` contient la valeur aberrante -1, qui ne correspond à aucune catégorie valide dans la documentation BAAC. Pour résoudre ce problème et garantir la cohérence des données, nous avons converti cette colonne en type entier (Int64), ce qui permet d’éliminer automatiquement les valeurs invalides et de ne conserver que les positions valides dans le véhicule.

In [115]:
print(usagers["place"].unique())
usagers["place"] = usagers["place"].replace(-1, np.nan)

[ 1. 10.  2.  4.  3.  5.  7.  9.  8.  6. nan]


In [116]:
usagers.describe(include='all')

,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
count,1.251870e+05,125187,125187,125187,125184.000000,125187.000000,125187.000000,122792.0,122608.0,125187.000000,125187.000000,125187.000000,125187.000000,125187.000000,125187,125187.000000
unique,NaN,125187,92654,45,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,13,NaN
top,NaN,203 988 581,155 721 557,A01,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,-1,NaN
freq,NaN,1,56,75443,NaN,NaN,NaN,<NA>,<NA>,NaN,NaN,NaN,NaN,NaN,61726,NaN
mean,2.024000e+11,NaN,NaN,NaN,2.100820,1.335554,2.524208,1.317024,1985.075411,3.069009,1.938788,0.843906,-0.800994,-0.223002,NaN,-0.814605
std,1.567431e+04,NaN,NaN,NaN,2.584527,0.610862,1.374424,0.465319,19.327486,2.762129,2.384028,2.910122,1.067398,1.276891,NaN,0.638087
min,2.024000e+11,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.0,1914.0,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,NaN,-1.000000
25%,2.024000e+11,NaN,NaN,NaN,1.000000,1.000000,1.000000,1.0,1971.0,0.000000,1.000000,-1.000000,-1.000000,-1.000000,NaN,-1.000000
50%,2.024000e+11,NaN,NaN,NaN,1.000000,1.000000,3.000000,1.0,1988.0,4.000000,1.000000,0.000000,-1.000000,0.000000,NaN,-1.000000
75%,2.024000e+11,NaN,NaN,NaN,2.000000,2.000000,4.000000,2.0,2001.0,5.000000,2.000000,0.000000,-1.000000,0.000000,NaN,-1.000000


In [120]:
usagers["actp"].isna().sum()

np.int64(0)

Plus de case manquante dans l'attribut `actp`

#### **4) Nettoyage de la table vehicules**

**4.1 Exploration Initiale de la table `vehicules`**

In [121]:
print("Les colonnes de la table vehicules :\n")
print(vehicules.columns)
print("\n")
print("Les différentes infos de la table :\n")
print(vehicules.info)
print("\n")
print("Apperçu de la table : \n")
vehicules.head()

Les colonnes de la table vehicules :

Index(['Num_Acc', 'id_vehicule', 'num_veh', 'senc', 'catv', 'obs', 'obsm',
       'choc', 'manv', 'motor', 'occutc'],
      dtype='str')


Les différentes infos de la table :

<bound method DataFrame.info of             Num_Acc  id_vehicule num_veh  senc  catv  obs  obsm  choc  manv  \
0      202400000001  155 781 758     A01     1     7    0     2     1    13   
1      202400000001  155 781 759     B01     2    14    0     2     2    21   
2      202400000002  155 781 757     A01     1    10    0     1     3    15   
3      202400000003  155 781 756     A01     3     3    9     1     1     1   
4      202400000004  155 781 754     B01     3    34    0     2     0     0   
...             ...          ...     ...   ...   ...  ...   ...   ...   ...   
92673  202400054401  155 686 119     Y01     1    60    1     2     7     0   
92674  202400054401  155 686 120     A01     1    33    1     2     1     1   
92675  202400054402  155 686 118     A01   

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
0,202400000001,155 781 758,A01,1,7,0,2,1,13,1,NaN
1,202400000001,155 781 759,B01,2,14,0,2,2,21,1,NaN
2,202400000002,155 781 757,A01,1,10,0,1,3,15,1,NaN
3,202400000003,155 781 756,A01,3,3,9,1,1,1,1,NaN
4,202400000004,155 781 754,B01,3,34,0,2,0,0,1,NaN


In [124]:
print("Le nombre de cellule vide dans chaque colonne : \n")
print(vehicules.isna().sum())
print("\n")
print("Le nombre de doublons dans la table :", +vehicules.duplicated().sum())


Le nombre de cellule vide dans chaque colonne : 

Num_Acc            0
id_vehicule        0
num_veh            0
senc               0
catv               0
obs                0
obsm               0
choc               0
manv               0
motor              0
occutc         91729
dtype: int64


Le nombre de doublons dans la table : 0


Il n’y a aucun doublon dans la table `vehicules`. En revanche, l’attribut `occutc` contient 91 729 cellules vides, ce qui indique un problème de complétude des données.
Pour mieux comprendre l’origine de ces valeurs manquantes, examinons maintenant les types de chaque attribut afin d’identifier d’éventuelles incohérences.

In [125]:
vehicules.dtypes

Num_Acc          int64
id_vehicule        str
num_veh            str
senc             int64
catv             int64
obs              int64
obsm             int64
choc             int64
manv             int64
motor            int64
occutc         float64
dtype: object

In [127]:
vehicules.describe(include='all')

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
count,9.267800e+04,92678,92678,92678.000000,92678.000000,92678.000000,92678.000000,92678.000000,92678.000000,92678.000000,949.000000
unique,NaN,92678,45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,155 781 758,A01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,1,53521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,2.024000e+11,NaN,NaN,1.611019,13.785418,1.034194,1.680345,2.870293,6.849101,1.370455,2.066386
std,1.567832e+04,NaN,NaN,0.819139,14.640063,3.135362,1.328128,2.418100,7.912183,1.096123,4.556816
min,2.024000e+11,NaN,NaN,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,-1.000000,0.000000
25%,2.024000e+11,NaN,NaN,1.000000,7.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000
50%,2.024000e+11,NaN,NaN,2.000000,7.000000,0.000000,2.000000,2.000000,2.000000,1.000000,1.000000
75%,2.024000e+11,NaN,NaN,2.000000,10.000000,0.000000,2.000000,4.000000,14.000000,1.000000,1.000000


Etant donnee que l'attribut `occutcp` est de type float les cases vides ne constituent pas une erreur. on peut cependant convertirl'attribut en int pour plus de coherence car ce code represente le nombre d’occupants dans le transport 

In [143]:
vehicules["occutc"] = vehicules["occutc"].astype("Int64")
print("Nouveau type de l'attribu occutc : ",vehicules["occutc"].dtypes)

Nouveau type de l'attribu occutc :  Int64


**Vérification des valeurs aberrantes dans la table `vehicules`**

In [132]:
vehicules["occutc"].unique()

array([nan,  1.,  2.,  4., 10.,  7.,  6.,  8.,  5.,  3., 13., 28., 43.,
       11., 32., 46.,  0.,  9., 51., 56., 12., 19., 47., 50.])

Selon la documentation, l’attribut `catv` doit contenir un code compris entre 0 et 99. La présence de la valeur -1 constitue donc une anomalie, car elle ne fait pas partie des valeurs autorisées. Nous remplaçons donc cette valeur aberrante par 0 afin d’harmoniser la colonne.

In [139]:
vehicules[vehicules["catv"] == -1]                  # Affichage de/des lignes ayant un occutc valant -1 

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
84280,202400049415,155 694 739,AB01,0,-1,0,0,0,0,-1,NaN


In [140]:
vehicules["catv"] = vehicules["catv"].replace(-1,0) # Remplacement de ou des elements -1 en 0
vehicules[vehicules["catv"] == -1]                  # Vérification de la modification

,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc


La valeur -1 a bien été supprimé 